#importing


In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

#loading dataset


In [48]:
data=pd.read_csv('data.csv')
data.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [49]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [50]:
data.drop(['Unnamed: 32', 'id'], axis=1, inplace=True)
data.diagnosis=[1 if each =="M" else 0 for each in data.diagnosis]

#input and output data

In [51]:
y= data.diagnosis.values
x_data=data.drop(['diagnosis'],axis=1)

#normalisation

In [52]:
x=(x_data-np.min(x_data)) / (np.max(x_data)-np.min(x_data))

#splitting data for training and testing

In [53]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.15,random_state=42) #test=15%

x_train=x_train.T
x_test=x_test.T
y_train=y_train.T
y_test=y_test.T

print("x train: ",x_train.shape)
print("x test: ",x_test.shape)
print("y train: ",y_train.shape)
print("y test: ",y_test.shape)


x train:  (30, 483)
x test:  (30, 86)
y train:  (483,)
y test:  (86,)


#weight bias

In [54]:
def intiallize_weight_bias(dimension):
  w=np.full((dimension,1),0.01)
  b=0.0
  return w,b

#sigmoid function-calculating z value

In [55]:
# z=np.dot(w,T,x_train)+b
def sigmoid(z):
  y_head=1/(1+np.exp(-z))
  return y_head

#forward-backward propagation

In [56]:
def forward_backward_propagation(w, b, x_train, y_train):
    z = np.dot(w.T, x_train) + b
    y_head = sigmoid(z)
    loss=y_train*np.log(y_head)+(1-y_train)*np.log(1-y_head)
    # x_train.shape[1] is for scaling
    cost=(np.sum(loss))/(x_train.shape[1])

    #backward propagation
    derivative_weight = (np.dot(x_train,((y_head-y_train).T)))/x_train.shape[1]
    derivative_bias = np.sum(y_head-y_train)/x_train.shape[1]

    gradients = {"derivative_weight": derivative_weight, "derivative_bias": derivative_bias}
    return cost, gradients

#updating Parameters

In [57]:
def update(w, b, x_train, y_train, learning_rate, num_of_iterations):
    cost_list = []
    cost_list2 = []
    index=[]
    # updating (learning) paramaters is number_of_iteration times
    for i in range(num_of_iterations):
      # make forward and backward propagation and find cost and gradients
        cost, gradients = forward_backward_propagation(w, b, x_train, y_train)
        cost_list.append(cost)
        # lets update
        w -= learning_rate * gradients["derivative_weight"]
        b -= learning_rate * gradients["derivative_bias"]

        if i % 100 == 0:
            cost_list2.append(cost)
            index.append(i)

            print(f"Cost after iteration %{i}: {cost}")

    # update (learn) parameters weights and bias
    parameters = {"weight": w, "bias": b}
    plt.plot(index, cost_list2)
    plt.xticks(index, rotation='vertical')
    plt.xlabel("Number of Iterarion")
    plt.ylabel("Cost")
    plt.show()
    return parameters, gradients, cost_list

#predictions

In [58]:
def predict(w, b, x_test):
  # x_test is a input for forward propagation
    z = sigmoid(np.dot(w.T, x_test) + b)
    y_prediction = np.zeros((1, x_test.shape[1]))

    # if z is higher than 0.5 , our prediction is sign one (y_head-1)
    # if z is smaller than 0.5 , our prediction is sign zero (y_head-0)

    for i in range(z.shape[1]):
        y_prediction[0, i] = 1 if z[0, i] > 0.5 else 0

    return y_prediction

#logistic regression

In [ ]:
from re import DEBUG
import numpy as np

def intialize_weight_and_bias(dimension):
  w=np.zeros((dimension ,1))
  b=0
  return w,b

def sigmoid(z):
    return 1/(1+np.exp(-z))

def propagate(w,b,x_train,y_train):
    m=x_train.shape[1]


    # forward propagation
    A=sigmoid(np.dot(w.T,x_train)+b)
    cost=-(1/m)*np.sum(y_train*np.log(A)+(1-y_train)*np.log(1-A))

    # backward propagation
    dw=(1/m)*np.dot(x_train,(A-y_train).T)
    db=(1/m)*np.sum(A-y_train)

    assert(dw.shape==w.shape)
    assert(db.dtype==float)
    cost=np.squeeze(cost)
    assert(cost.shape==())

    return dw,db,cost
def update(w,b,X,Y,learning_rate,num_iterations):
  cost_list=[]

  for i in range(num_iterations):
    dw,db,cost=propagate(w,b,X,Y)
    w-=learning_rate*dw
    b-=learning_rate*db

    if i%100==0:
      cost_list.append(cost)

    if np.isnan(cost): #check for NaN values
        print(f"cost is NaN at iteration {i}")
        break

  parameters={"weight":w,"bias":b}
  gradients={"dw":dw,"db":db}
  return parameters,gradients,cost_list

def predict (w,b,X):
  m=X.shape[1]
  Y_prediction=np.zeros((1,m))
  A=sigmoid(np.dot(w.T,X)+b)

  for i in range(A.shape[1]):
    Y_prediction[0,i]=1 if A[0,i]>0.5 else 0


  return Y_prediction

def logistic_regression(X_train,Y_train,X_test,Y_test,learning_rate,num_iterations):
  dimension=X_train.shape[0]
  w,b=intialize_weight_and_bias(dimension)

  parameters,gradients,cost_list=update(w,b,x_train,y_train,learning_rate,num_iterations)

  y_prediction_test=predict(parameters["weight"],parameters["bias"],x_test)
  y_prediction_train=predict(parameters["weight"],parameters["bias"],x_train)

  #train/test errors
  print("train accuracy: {} %".format(100 - np.mean(np.abs(y_prediction_train - y_train)) * 100))
  print("test accuracy: {} %".format(100 - np.mean(np.abs(y_prediction_test - y_test)) * 100))


# example data (replace with your acutal data)\
x_train=np.random.randn(10,100)
y_train=np.random.randint(0,2,(1,100))
x_test=np.random.randn(10,20)
y_test=np.random.randint(0,2,(1,20))

logistic_regression(x_train,y_train,x_test,y_test,learning_rate=.01,num_iterations=1000)



#check results with linear_model.LogisticRegression

In [60]:
from sklearn import linear_model
logreg=linear_model.LogisticRegression(random_state=42,max_iter=150)
print("test accuracy:{} ".format(logreg.fit(x_train.T,y_train.T).score(x_test.T,y_test.T)))
print("train accuracy:{} ".format(logreg.fit(x_train.T,y_train.T).score(x_train.T,y_train.T)))

test accuracy:0.45 
train accuracy:0.62 
